# 📊 Notebook 06 · Informe Ejecutivo
---
**Proyecto:** Bogotá — Análisis Predictivo del Mercado Inmobiliario Urbano  
**Fuente:** UAECD / IDECA · 2020–2026  
**Autor:** Kevin Palacio Martinez  

---
## 🎯 Objetivo

Sintetizar los hallazgos de las cinco fases del proyecto en un informe ejecutivo,
identificar zonas de mayor oportunidad inmobiliaria y definir la hoja de ruta
de próximos pasos.

## 📋 Contenido

| Sección | Descripción | Notebooks fuente |
|---|---|---|
| 1 | Hallazgos descriptivos y temporales | NB 01 · 03 |
| 2 | Hallazgos espaciales | NB 02 |
| 3 | Evaluación de modelos predictivos | NB 04 · 04b |
| 4 | Expansión urbana — proyección 2030 | NB 05 |
| 5 | Zonas de mayor oportunidad | NB 02 · 04b · 05 |
| 6 | Limitaciones metodológicas | Transversal |
| 7 | Próximos pasos | — |

In [ ]:
import sys
sys.path.append('../src')

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Image
from sqlalchemy import create_engine
from db_config import DB_URL

pd.set_option('display.float_format', '{:,.0f}'.format)
sns.set_theme(style="whitegrid", font_scale=1.1)

engine = create_engine(DB_URL)

df              = pd.read_parquet('../data/processed/valor_manzana_limpio.parquet')
serie           = pd.read_parquet('../data/processed/serie_localidad.parquet')
df_valorizacion = pd.read_parquet('../data/processed/valorizacion_manzana.parquet')

print(f"✓ valor_manzana_limpio  : {len(df):,} registros · {df['cod_manzana'].nunique():,} manzanas")
print(f"✓ serie_localidad       : {len(serie):,} registros · {serie['localidad'].nunique()} localidades")
print(f"✓ valorizacion_manzana  : {len(df_valorizacion):,} manzanas con serie 2020–2026")

---
## 📈 Sección 1 · Hallazgos descriptivos y temporales

> **Base de datos:** 299,482 registros · 44,100 manzanas únicas · 20 localidades · 2020–2026  
> Fuentes: Notebooks 01 y 03.

In [ ]:
# ── Evolución de la mediana del valor por m² ──────────────────

resumen_anual = (
    df[df['valor_final'] > 0]
    .groupby('anio')['valor_final']
    .agg(mediana='median', registros='count')
    .reset_index()
)
resumen_anual['var_pct'] = resumen_anual['mediana'].pct_change().mul(100).round(2)

med_2020 = resumen_anual.loc[resumen_anual['anio'] == 2020, 'mediana'].values[0]
med_2026 = resumen_anual.loc[resumen_anual['anio'] == 2026, 'mediana'].values[0]
valorizacion_acum = (med_2026 - med_2020) / med_2020 * 100

# IPC acumulado 2021-2025 (DANE)
ipc_acum = (1.0562 * 1.1312 * 1.0928 * 1.052 * 1.051) - 1
val_real = valorizacion_acum - ipc_acum * 100

display(
    resumen_anual.rename(columns={
        'anio':     'Año',
        'mediana':  'Mediana (COP/m²)',
        'registros':'Registros',
        'var_pct':  'Var. anual (%)'
    })
)

print(f"── Indicadores clave 2020–2026 ──────────────────────────")
print(f"  Valorización nominal acumulada  : +{valorizacion_acum:.1f}%")
print(f"  IPC acumulado 2021–2025 (DANE)  :  {ipc_acum*100:.1f}%")
print(f"  Valorización real estimada      : +{val_real:.1f}%")
print(f"  Mediana 2020 : ${med_2020:,.0f} COP/m²")
print(f"  Mediana 2026 : ${med_2026:,.0f} COP/m²")

# ── Tabla consolidada de hallazgos ────────────────────────────
hallazgos = pd.DataFrame([
    (1,  "Descriptivo", "Valorización acumulada urbana",           "+59%  ·  $1.625M → $2.600M / m²"),
    (2,  "Descriptivo", "Año de mayor aceleración",                "2026  ·  +13% anual"),
    (3,  "Descriptivo", "Homogeneización urbana",                  "CV: 67% → 60%  ·  zonas rezagadas convergen"),
    (4,  "Descriptivo", "Compresión brecha urbano / periurbano",   "386% → 273%  ·  presión de expansión"),
    (5,  "Descriptivo", "Periurbano lidera valorización desde 2023","Supera el promedio urbano anual"),
    (6,  "Temporal",    "Valorización real positiva vs IPC",       "Suelo supera la inflación en la mayoría de años"),
    (7,  "Temporal",    "Quiebre estructural 2022",                "Desaceleración  ·  tasas BanRep → 13.25%"),
    (8,  "Temporal",    "Reactivación 2026",                       "Ciclo de recorte de tasas  ·  +13% anual"),
    (9,  "Temporal",    "Heterogeneidad entre localidades",        "Brecha de hasta 3× en valorización acumulada"),
], columns=["#", "Fase", "Hallazgo", "Detalle"])

print("\n── Tabla consolidada de hallazgos descriptivos y temporales ─")
display(hallazgos)

display(Image('../reports/figures/01_evolucion_valor_anual.png'))
display(Image('../reports/figures/10_valorizacion_vs_ipc.png'))

---
## 🗺️ Sección 2 · Hallazgos espaciales

> Resultados de autocorrelación espacial y clusters LISA — Notebook 02.  
> Índice de Moran's I calculado sobre muestra de 8,000 manzanas por año.  
> Clusters LISA calculados con muestra de 6,000 manzanas.

In [ ]:
# ── Hallazgos espaciales consolidados ────────────────────────

hallazgos_esp = pd.DataFrame([
    (1, "Autocorrelación",   "Moran's I positivo en todos los años",        "Valores similares se agrupan · no hay distribución aleatoria"),
    (2, "Cluster HH",        "Chapinero · Teusaquillo · Barrios Unidos",    "Núcleos premium · alto valor estable"),
    (3, "Cluster LL",        "Ciudad Bolívar · Usme · San Cristóbal",       "Zonas rezagadas · potencial a largo plazo"),
    (4, "Cluster LH",        "Periurbano con vecindad de alto valor",       "Bajo valor rodeado de alto  ·  señal de presión expansiva"),
    (5, "Territorial 2026",  "41,862 manzanas urbanas",                     "1,724 periurbanas  ·  1,362 rurales"),
    (6, "Brecha territorial","Valor periurbano ≈ 27% del urbano",           "Brecha 273%  ·  comprimida desde 386% en 2020"),
], columns=["#", "Categoría", "Hallazgo", "Detalle"])

display(hallazgos_esp)

display(Image('../reports/figures/05_morans_i_evolucion.png'))
display(Image('../reports/figures/06_lisa_clusters_2020_2026.png'))

---
## 🤖 Sección 3 · Evaluación de modelos predictivos

> Resultados de la comparación de 10 modelos — Notebooks 04 y 04b.  
> **Target:** valor comercial por m² a nivel de manzana.  
> **Dataset:** 252,700 registros · 9 features · split 80/20.

In [ ]:
# ── Ranking de modelos ─────────────────────────────────────────

ruta_comp = '../data/processed/comparacion_modelos_04b.csv'
if os.path.exists(ruta_comp):
    df_modelos = pd.read_csv(ruta_comp)
    df_validos = (
        df_modelos[df_modelos['Estado'] == '✓']
        .dropna(subset=['R²'])
        .sort_values('R²', ascending=False)
        .reset_index(drop=True)
    )
    print("── Ranking de 10 modelos evaluados ──────────────────────")
    display(df_validos[['Modelo', 'R²', 'MAE', 'RMSE', 'Tiempo (s)']])
else:
    print("⚠ comparacion_modelos_04b.csv no encontrado")

# ── Modelo seleccionado: XGBoost ──────────────────────────────
print("\n── Modelo seleccionado para producción ──────────────────")
resumen_ganador = pd.DataFrame([{
    'Modelo':           'XGBoost',
    'R²':               0.8454,
    'MAE':              '$308,368 COP/m²',
    'RMSE':             '$557,483 COP/m²',
    'Tipo':             'Gradient Boosting',
    'Features':         9,
    'Registros train':  '202,160',
    'Registros test':   '50,540',
}])
display(resumen_ganador)

# ── Importancia de variables (SHAP / RF) ──────────────────────
print("\n── Importancia de variables ─────────────────────────────")
importancia = pd.DataFrame([
    ('estrato',             'Alta',   'Variable socioeconómica de mayor peso · +$767K por unidad'),
    ('dist_metro_m',        'Alta',   'Cercanía al metro incrementa valor  ·  -$249K por km'),
    ('dist_tm_m',           'Alta',   'Acceso a TransMilenio  ·  -$165K por km'),
    ('dist_sitp_m',         'Media',  'Cobertura SITP  ·  -$42K por km'),
    ('indice_inseguridad',  'Media',  'Impacto negativo en zonas de alto índice'),
    ('anio_norm',           'Media',  'Tendencia temporal  ·  +$225K por año'),
    ('territorio_enc',      'Baja',   'Urbano vs periurbano'),
    ('variacion_localidad', 'Baja',   'Momentum de valorización local'),
    ('localidad_enc',       'Baja',   'Efecto fijo de localidad'),
], columns=['Variable', 'Importancia', 'Interpretación'])
display(importancia)

# ── Distribución de errores del modelo ganador ────────────────
print("\n── Calidad de predicciones XGBoost (manzanas 2026) ─────")
errores = pd.DataFrame([
    ('Excelente',  '<10% error',    20_462, '49.2%'),
    ('Bueno',      '10–25% error',  12_289, '29.6%'),
    ('Moderado',   '25–50% error',   4_831, '11.6%'),
    ('Alto',       '50–100% error',  1_683,  '4.1%'),
    ('Muy alto',   '>100% error',    2_263,  '5.4%'),
], columns=['Nivel', 'Rango', 'Manzanas', '% del total'])
display(errores)
print("  ✓ El modelo predice con error < 25% en el 78.8% de las manzanas")

display(Image('../reports/figures/18_comparacion_todos_modelos.png'))
display(Image('../reports/figures/16b_shap_summary.png'))

---
## 🌱 Sección 4 · Expansión urbana — Proyección 2030

> Resultados del autómata celular — Notebook 05.  
> El modelo simula la transición de manzanas entre estados (Rural → Periurbano → Urbano)  
> a partir de variables de proximidad, uso de suelo y tratamientos POT.  
>
> **Base:** 41,823 manzanas con serie completa 2020–2026.

In [ ]:
# ── Proyección de estados territoriales 2026–2030 ─────────────

proyeccion = pd.DataFrame([
    (2026, '41,823 (base)', '—',    '—',    'Estado inicial'),
    (2027, '41,578',        '123',  '122',  'Primera transición'),
    (2028, '41,623',         '99',  '101',  'Consolidación'),
    (2029, '41,658',         '84',   '81',  'Estabilización'),
    (2030, '41,681',         '73',   '69',  'Horizonte final'),
], columns=['Año', 'Urbano', 'Periurbano', 'Rural', 'Nota'])

print("── Proyección del estado territorial 2026–2030 ──────────")
display(proyeccion)

print("\n── Manzanas con expansión proyectada 2026 → 2030 ────────")
print("  Total con expansión    : 290 manzanas")
print("  Rural → Periurbano     :  55 manzanas  (19%)")
print("  Periurbano → Urbano    : 212 manzanas  (73%)")
print("  Sin cambio             : 41,533 manzanas")

transiciones = pd.DataFrame([
    ('Rural → Periurbano',    55, '19.0%', 'Zonas en incorporación activa · franja sur y suroccidente'),
    ('Periurbano → Urbano',  212, '73.1%', 'Franja periurbana consolidándose · mayor magnitud'),
    ('Sin cambio',        41_556,   '—',   'Estado estable en el horizonte 2030'),
], columns=['Transición', 'Manzanas', '% del total con expansión', 'Interpretación'])
display(transiciones)

# ── Gráfico de la proyección ──────────────────────────────────
años    = [2027, 2028, 2029, 2030]
periurb = [123,   99,   84,   73]
rural   = [122,  101,   81,   69]
urbano  = [41_578, 41_623, 41_658, 41_681]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1 — Manzanas en transición
axes[0].plot(años, periurb, marker='o', linewidth=2.5, color='#F59E0B',
             markersize=8, label='Periurbano')
axes[0].plot(años, rural,   marker='s', linewidth=2.5, color='#16A34A',
             markersize=8, label='Rural')
axes[0].fill_between(años, periurb, alpha=0.15, color='#F59E0B')
axes[0].fill_between(años, rural,   alpha=0.15, color='#16A34A')
axes[0].set_title('Manzanas en zona de transición\n(de 41,823 totales)', fontsize=11)
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Número de manzanas')
axes[0].legend()
axes[0].set_xticks(años)

# Panel 2 — Nuevas urbanas acumuladas desde 2027
nuevas = [u - 41_578 for u in urbano]
axes[1].bar(años, nuevas, color='#2563EB', alpha=0.85, edgecolor='white')
axes[1].set_title('Nuevas manzanas incorporadas\nal tejido urbano (desde 2027)', fontsize=11)
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Manzanas acumuladas desde 2027')
axes[1].set_xticks(años)
for i, v in enumerate(nuevas):
    axes[1].text(años[i], v + 0.3, str(v), ha='center', fontsize=10, fontweight='bold')

# Panel 3 — Tipo de transición
bars = axes[2].bar(
    ['Rural →\nPeriurbano', 'Periurbano →\nUrbano'],
    [55, 212],
    color=['#16A34A', '#2563EB'], alpha=0.85, edgecolor='white'
)
axes[2].set_title('Tipo de transición proyectada\n2026 → 2030', fontsize=11)
axes[2].set_ylabel('Número de manzanas')
for bar, val in zip(bars, [55, 212]):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 1,
                 str(val), ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/24_resumen_expansion_2030.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado → reports/figures/24_resumen_expansion_2030.png")

display(Image('../reports/figures/23_automata_celular_2030.png'))

---
## 🎯 Sección 5 · Zonas de mayor oportunidad

> Identificación de zonas con mayor potencial de valorización, combinando tres señales:
>
> | Señal | Descripción | Fuente |
> |---|---|---|
> | **Subvaloración modelo** | `valor_predicho > valor_actual` según XGBoost | NB 04b |
> | **Expansión proyectada** | Manzanas periurbanas que transitarán a urbano antes de 2030 | NB 05 |
> | **Presión vecinal** | Clusters LISA tipo LH (bajo valor rodeado de alto valor) | NB 02 |

In [ ]:
# ── Señal 1 · Subvaloración según XGBoost ─────────────────────

query_sub = """
    SELECT
        cod_manzana, localidad, sector, tipo_territorio,
        valor_final, valor_predicho, diferencia, error_pct, estrato,
        dist_tm_m, dist_metro_m
    FROM catastro.predicciones_xgboost_2026
    WHERE tipo_territorio IN ('urbano', 'periurbano')
"""
df_pred = pd.read_sql(query_sub, engine)
df_subvaloradas = df_pred[df_pred['diferencia'] > 0].copy()
df_subvaloradas['gap_pct'] = (
    df_subvaloradas['diferencia'] / df_subvaloradas['valor_final'] * 100
).round(1)

print(f"▸ Manzanas con valor predicho > valor actual : {len(df_subvaloradas):,}")
print(f"  Periurbano : {(df_subvaloradas['tipo_territorio'] == 'periurbano').sum():,}")
print(f"  Urbano     : {(df_subvaloradas['tipo_territorio'] == 'urbano').sum():,}")

# ── Ranking de subvaloración por localidad ────────────────────
ranking_loc = (
    df_subvaloradas.groupby('localidad')
    .agg(
        manzanas=('cod_manzana', 'count'),
        gap_mediano_pct=('gap_pct', 'median'),
        valor_actual_med=('valor_final', 'median'),
        valor_predicho_med=('valor_predicho', 'median')
    )
    .sort_values('manzanas', ascending=False)
    .reset_index()
)
print("\n── Top 10 localidades con más manzanas subvaloradas ─────")
display(ranking_loc.head(10))

# ── Señal combinada: Periurbano + subvaloración ────────────────
periurbano_subval = df_subvaloradas[
    df_subvaloradas['tipo_territorio'] == 'periurbano'
].copy()

print(f"\n▸ Periurbano subvalorado (Señal 1 + Señal 2 combinadas) : {len(periurbano_subval):,}")
print(f"  Diferencia mediana  : ${periurbano_subval['diferencia'].median():,.0f} COP/m²")
print(f"  Gap % mediano       : {periurbano_subval['gap_pct'].median():.1f}%")

top_oportunidad = (
    periurbano_subval[
        ['localidad', 'sector', 'valor_final', 'valor_predicho', 'diferencia', 'gap_pct', 'estrato']
    ]
    .sort_values('diferencia', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print("\n── Top 15 manzanas periurbanas con mayor señal de oportunidad ─")
display(top_oportunidad)

# ── Visualización ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1 — Ranking de subvaloración por localidad
top10 = ranking_loc.head(10).sort_values('manzanas')
axes[0].barh(
    top10['localidad'], top10['manzanas'],
    color='#F59E0B', alpha=0.85, edgecolor='white'
)
axes[0].set_title('Manzanas subvaloradas por localidad\n(valor predicho > valor actual · XGBoost 2026)',
                  fontsize=11)
axes[0].set_xlabel('Número de manzanas')

# Panel 2 — Gap % por tipo de territorio
gap_tipo = (
    df_subvaloradas.groupby('tipo_territorio')['gap_pct']
    .median()
    .reset_index()
)
colores_tipo = {'periurbano': '#F59E0B', 'urbano': '#2563EB'}
axes[1].bar(
    gap_tipo['tipo_territorio'],
    gap_tipo['gap_pct'],
    color=[colores_tipo[t] for t in gap_tipo['tipo_territorio']],
    alpha=0.85, edgecolor='white'
)
axes[1].set_title('Gap de subvaloración mediano\npor tipo de territorio (%)',fontsize=11)
axes[1].set_ylabel('(Valor predicho - Valor actual) / Valor actual  (%)')
for i, (_, row) in enumerate(gap_tipo.iterrows()):
    axes[1].text(i, row['gap_pct'] + 0.3,
                 f"{row['gap_pct']:.1f}%", ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/25_zonas_oportunidad.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado → reports/figures/25_zonas_oportunidad.png")

display(Image('../reports/figures/22_clusters_expansion.png'))

---
## ⚠️ Sección 6 · Limitaciones del análisis

### Limitaciones de datos

| Limitación | Descripción | Impacto |
|---|---|---|
| **Cobertura rural limitada** | Solo 1,362 manzanas rurales con valor catastral registrado | Sumapaz excluida; análisis rural incompleto |
| **Ruptura en serie 2022–2023** | El salto 9 → 191 manzanas rurales refleja actualización de capa UAECD, no cambio real | Comparaciones interanuales en zona rural no son fiables |
| **Valores en cero** | 5,663 registros (1.9%) con valor = 0 en 2020, 2021 y 2025 | Requirió imputación por mediana de sector-año |
| **Manzanas sin localidad** | 5 manzanas únicas sin intersección con polígono de localidad | 27 registros excluidos del análisis de localidad |
| **IPC 2026 no disponible** | La variación IPC para 2026 no está publicada al cierre | Valorización real 2026 es estimada |

### Limitaciones metodológicas

| Limitación | Descripción | Impacto |
|---|---|---|
| **Moran's I — muestra** | Calculado sobre muestra de 8,000 manzanas por año | Estimación global, no censal |
| **LISA — ~30% sin cluster** | Muestra de 6,000 manzanas; el resto etiquetado "Sin calcular" | Cobertura parcial del análisis LISA |
| **OLS espacial en lugar de GWR** | Se usó spreg OLS con diagnósticos, no GWR propiamente dicho | Coeficientes globales, no locales por manzana |
| **Autómata celular determinístico** | Probabilidades de transición estáticas | Las proyecciones 2030 asumen continuidad del patrón actual |
| **Serie histórica corta** | Solo 7 años disponibles (2020–2026) | Modelos de series de tiempo con profundidad histórica limitada |
| **XGBoost sin validación temporal** | El split fue aleatorio (80/20), no cronológico | Posible fuga de información entre períodos |

---
## 🚀 Sección 7 · Próximos pasos

### Mejoras de datos

| Prioridad | Acción | Beneficio |
|---|---|---|
| Alta | Incorporar escrituras notariales SNR | Validar valores catastrales con precios de mercado reales |
| Alta | Agregar equipamientos urbanos IDECA (parques · colegios · hospitales) | Enriquecer features del modelo |
| Alta | Índice de valorización por UPZ | Mayor granularidad espacial |
| Media | Integrar normativa POT completa por manzana | Fundamental para proyecciones de expansión |
| Media | Actualizar capa microterritorios rurales UAECD 2025 | Corregir ruptura metodológica 2022–2023 |

### Mejoras de modelado

| Prioridad | Acción | Beneficio esperado |
|---|---|---|
| Alta | Reentrenar XGBoost con Optuna / Hyperopt | R² 0.845 → 0.88+ |
| Alta | Implementar GWR real (mgwr) para coeficientes locales | Mapas de efecto por variable en cada zona |
| Media | Modelo de precios hedónicos con variables instrumentales | Causalidad, no solo correlación |
| Media | Calibrar autómata celular con expansión histórica real | Reducir incertidumbre en proyecciones 2030 |
| Baja | Explorar TabNet con embeddings espaciales | Capturar interacciones complejas de vecindad |

### Productos derivados

| Producto | Descripción | Estado |
|---|---|---|
| Dashboard Power BI | Valor m² · predicciones · expansión proyectada por manzana | Pendiente |
| API REST | Endpoint de predicción de valor dado un conjunto de features | Pendiente |
| Mapa web (Folium / Leaflet) | Mapa interactivo público con zonas de oportunidad | Parcialmente disponible |
| Nota técnica | Documento metodológico para ACOLFA / ODEI / Secretaría Hábitat | Pendiente |

---
## ✅ Cierre — Bogotá · Análisis Predictivo del Mercado Inmobiliario Urbano

### Síntesis ejecutiva

Análisis completo del mercado inmobiliario urbano de Bogotá para el período **2020–2026**,
con proyección de expansión urbana al **2030**.

### 📊 Métricas finales del proyecto

| Indicador | Valor |
|---|---|
| Registros analizados | 299,482 |
| Manzanas únicas | 44,100 |
| Período de análisis | 2020–2026 |
| Localidades cubiertas | 19 (excluye Sumapaz) |
| Modelos evaluados | 10 |
| Modelo seleccionado | XGBoost |
| R² modelo final | 0.8454 |
| MAE modelo final | $308,368 COP/m² |
| Predicciones con error < 25% | 78.8% de las manzanas |
| Manzanas con expansión proyectada al 2030 | 290 |
| Periurbano → Urbano | 212 manzanas |
| Rural → Periurbano | 55 manzanas |

### 📦 Artefactos generados

| Carpeta | Archivos principales |
|---|---|
| `data/processed/` | `valor_manzana_limpio.parquet` · `serie_localidad.parquet` · `valorizacion_manzana.parquet` · `predicciones_rf.parquet` |
| `models/` | `xgboost_final_v1.joblib` · `random_forest_v1.joblib` · `modelo_final_metadata.json` |
| `reports/figures/` | 25 figuras estáticas PNG · 7+ mapas interactivos HTML |

---
*Proyecto: Bogotá — Análisis Predictivo del Mercado Inmobiliario Urbano*  
*Fuente: UAECD / IDECA · DANE · Banco de la República · SDM Bogotá*  
*Python 3.12 · PostGIS 3.6 · scikit-learn · xgboost · lightgbm · geopandas*  
*Autor: Kevin Palacio Martinez · kpalaciom@unal.edu.co*